# XGBoost Multiclass Classification

## Imports

In [43]:
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from collections import Counter


## Load Dataset

In [44]:
# Load dataset
data = pd.read_csv('T49.2_Sep2025_1_StGallen.csv', sep=';', na_values=["", " ", "NA", "N/A", "NULL"])
display(data.head(30))

threshold = 0.75
data = data.loc[:, data.isnull().mean() < threshold]

binary_mode = True
if binary_mode:
    data = data[data["OUTCOME_3Kat_KHK"] != 1].reset_index(drop=True)


/var/folders/m3/k3b3svlj7tvdqbtttg5w9xfw0000gn/T/ipykernel_22953/3323366592.py:2: DtypeWarning: Columns (3,32,119,121,125,129,231,330,342,588) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('T49.2_Sep2025_1_StGallen.csv', sep=';', na_values=["", " ", "NA", "N/A", "NULL"])


,OUTCOME_3Kat_KHK,Alter_0,geschlecht,waist_0,WHR_0,BMI_0,currsmo0,RR_syst_0,RR_diast_0,Pulse_Pressure,...,SMC231l,SMC232l,SMC233l,SMC240l,SMC241l,SMC242l,SMC243l,SMC244l,SMC263l,SMC264l
0,2,"56,9691991786448",0,112,",982456140350877","32,8731097961867",0,130.0,80.0,50.0,...,"9,44","4,38","3,99","14,1","25,8","7,8","3,99","3,99","3,99","3,99"
1,2,"69,4976043805613",0,104,"1,04","30,1102788964583",0,160.0,70.0,90.0,...,"19,8","8,97","7,75","21,9","41,3","20,1","7,35",NaN,"7,45","8,11"
2,2,"72,0492813141684",0,129,",941605839416058","44,7348800491207",0,140.0,90.0,50.0,...,"13,9","4,75","3,99","20,3","52,4","18,6","3,99","3,99","3,99","3,99"
3,2,"70,8145106091718",0,99,",876106194690266","26,2595847484332",0,130.0,90.0,40.0,...,20,"8,73","7,85","25,3","50,7",17,0,NaN,"7,63","9,02"
4,2,"62,1601642710472",1,125,NaN,"42,4366343891054",1,180.0,100.0,80.0,...,"16,9","9,14","8,3","29,5",43,"17,9","8,13",0,"8,4","8,51"
5,2,"73,82340862423",1,97,",989795918367347","26,4462809917355",0,157.0,85.0,72.0,...,"13,1",8,"7,4","15,1","26,4","15,2","7,4","3,89","7,53","8,15"
6,2,"58,4284736481862",1,92,",978723404255319","25,78125",1,130.0,85.0,45.0,...,"16,5","8,61",0,"21,5",48,"20,4",0,NaN,"7,71","9,31"
7,2,"54,5954825462012",0,NaN,NaN,"32,4661454995942",1,140.0,80.0,60.0,...,"18,7","8,84","8,54","28,2","42,7","15,9","8,28",0,"8,44","8,83"
8,2,"76,1724845995893",0,92,",884615384615385","27,2392244832559",0,150.0,110.0,40.0,...,"18,8","8,84","7,71","18,6","40,4","20,8","7,03",0,"7,77","8,3"
9,2,"46,7953456536619",1,81,",931034482758621","20,8209399167162",1,110.0,70.0,40.0,...,"9,25","4,13","3,99","15,7","39,9","14,5","3,99","3,99","3,99","3,99"


## Preprocessing

In [45]:
# Split data into features and target
X = data.drop("OUTCOME_3Kat_KHK", axis=1)
y = data['OUTCOME_3Kat_KHK']
y = y.replace({2: 1})


# Define categorical and numerical features
categorical_features = X.select_dtypes(
   include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
   include=["float64", "int64"]
).columns.tolist()

# Convert categorical features to string type
X[categorical_features] = X[categorical_features].astype(str)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
   X, y, test_size=0.2, random_state=42
)



cat_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),   # fill missing categoricals
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

# Define preprocessor

preprocessor = ColumnTransformer(
   transformers=[
       ("cat", cat_pre, categorical_features),
       ("num", SimpleImputer(strategy="median"), numerical_features)
   ]
)

print("Before SMOTE:", Counter(y_train))

# transform X_train into numeric form before SMOTE
X_train_prep = preprocessor.fit_transform(X_train)

sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_train_prep, y_train)

print("After SMOTE:", Counter(y_resampled))

Before SMOTE: Counter({1: 1036, 0: 306})
After SMOTE: Counter({1: 1036, 0: 1036})


## Build Pipeline

In [46]:


# Create pipeline
pipeline = Pipeline(
   [
       ("preprocessor", preprocessor),
       ("smote", SMOTE(random_state=42)),
       ("classifier", XGBClassifier(
    tree_method="hist",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)),

   ]
)

#CV and training
# Perform 5-fold cross-validation

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

# Fit the model on the training data
pipeline.fit(X_train, y_train)

# Predict on the test set
y_pred = pipeline.predict(X_test)




## Evaluation

In [47]:

# Generate classification report
report = classification_report(y_test, y_pred)

print(f"Mean Cross-Validation Accuracy: {cv_scores.mean():.4f}")
print("\nClassification Report:")
print(report)


Mean Cross-Validation Accuracy: 0.8010

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.40      0.49        57
           1       0.89      0.95      0.92       279

    accuracy                           0.86       336
   macro avg       0.76      0.68      0.71       336
weighted avg       0.84      0.86      0.85       336



# XGBoost Binary Classification